# Who Says Yes, and When Does It Matter?
### Statistical Analysis of a Portuguese Bank's Direct Marketing Campaigns

**Dataset:** UCI Bank Marketing — `bank-additional-full.csv`  
**Records:** 41,188 phone call contacts | **Features:** 20 | **Period:** May 2008 – November 2010  
**Target:** Did the client subscribe to a term deposit? (`y`: yes/no)

---

The bank made over 41,000 phone calls. Only **11.3%** resulted in a subscription.  
This analysis asks: *why does the campaign underperform, and where are the bright spots?*

We use five statistical tests and three interactive visualizations to find out.

In [5]:
# ── Dependencies ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('bank-additional-full.csv', sep=';')
df['subscribed'] = (df['y'] == 'yes').astype(int)

print(f"Loaded {len(df):,} records × {df.shape[1]} features")
print(f"Subscription rate: {df['subscribed'].mean()*100:.1f}%")

Loaded 41,188 records × 22 features
Subscription rate: 11.3%


---
## Section 1 — The Imbalance Problem

Before any test, we need to understand the target distribution.  
An 88/12 split means a naive model that always predicts "no" would be 88% accurate — and completely useless.

In [6]:
counts = df['y'].value_counts()
pct = df['y'].value_counts(normalize=True) * 100

fig = go.Figure(go.Pie(
    labels=['No (did not subscribe)', 'Yes (subscribed)'],
    values=counts.values,
    hole=0.55,
    marker_colors=['#EF553B', '#00CC96'],
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Count: %{value:,}<br>Share: %{percent}<extra></extra>'
))
fig.update_layout(
    title=dict(text='Target Distribution — Term Deposit Subscriptions', font_size=16),
    annotations=[dict(text=f'{counts["yes"]:,}<br>subscribed', x=0.5, y=0.5,
                      font_size=14, showarrow=False)],
    height=420
)
fig.show()

---
## Section 2 — Does Call Duration Predict Subscription?

**Test: Mann-Whitney U** (non-parametric; duration is heavily right-skewed)

**Important caveat:** `duration` is flagged in the dataset documentation as a data leakage risk.  
A 0-second call means definite rejection; a long call suggests genuine engagement.  
In production you wouldn't know duration *before* the call — so it's not actionable as a predictor.  
We test it anyway because the *magnitude* of the difference tells us something real about what engagement looks like.

In [7]:
yes_dur = df[df['y'] == 'yes']['duration']
no_dur  = df[df['y'] == 'no']['duration']

u_stat, p_val = stats.mannwhitneyu(yes_dur, no_dur, alternative='greater')

print("── Mann-Whitney U: Duration (subscribers vs non-subscribers) ──")
print(f"  Subscribers   — median: {yes_dur.median():.0f}s  mean: {yes_dur.mean():.0f}s")
print(f"  Non-subscribers — median: {no_dur.median():.0f}s  mean: {no_dur.mean():.0f}s")
print(f"  U-statistic: {u_stat:,.0f}")
print(f"  p-value: {p_val:.2e}")
print(f"  Result: {'Significant ✓' if p_val < 0.05 else 'Not significant'}")
print(f"\n  Subscribers stay on the phone {yes_dur.median()/no_dur.median():.1f}× longer (median)")

# Violin plot
plot_df = df[df['duration'] < 1500].copy()  # trim extreme outliers for viz
fig = px.violin(
    plot_df, x='y', y='duration', color='y',
    color_discrete_map={'yes': '#00CC96', 'no': '#EF553B'},
    box=True, points=False,
    labels={'y': 'Subscribed', 'duration': 'Call Duration (seconds)'},
    title='Call Duration Distribution by Subscription Outcome',
    hover_data=['age', 'job']
)
fig.update_layout(height=450, showlegend=False)
fig.add_annotation(x=0.5, y=1.05, xref='paper', yref='paper',
    text=f'Mann-Whitney U p-value: {p_val:.2e} — highly significant',
    showarrow=False, font_size=11, font_color='gray')
fig.show()

── Mann-Whitney U: Duration (subscribers vs non-subscribers) ──
  Subscribers   — median: 449s  mean: 553s
  Non-subscribers — median: 164s  mean: 221s
  U-statistic: 138,794,276
  p-value: 0.00e+00
  Result: Significant ✓

  Subscribers stay on the phone 2.7× longer (median)


---
## Section 3 — The Economic Window: Interest Rates & Subscription Timing

The dataset spans May 2008 to November 2010 — covering the onset of the **2008 global financial crisis**.  
The `euribor3m` variable (European interbank lending rate) is our proxy for economic conditions.

**Hypothesis:** Subscription rates are higher when interest rates are lower — people are more willing to lock money into a term deposit when the broader economy is uncertain or rates have fallen.

**Test: Spearman rank correlation** between monthly euribor3m and monthly subscription rate.

In [8]:
month_order = ['mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
month_labels = {'mar':'Mar','apr':'Apr','may':'May','jun':'Jun',
                'jul':'Jul','aug':'Aug','sep':'Sep','oct':'Oct','nov':'Nov','dec':'Dec'}

monthly = (
    df.groupby('month')
    .agg(total=('y','count'),
         subscribed=('subscribed','sum'),
         euribor=('euribor3m','mean'))
    .reindex(month_order).dropna()
)
monthly['rate'] = monthly['subscribed'] / monthly['total'] * 100
monthly['month_label'] = monthly.index.map(month_labels)

rho, p_spear = stats.spearmanr(monthly['euribor'], monthly['rate'])
print("── Spearman Correlation: euribor3m vs monthly subscription rate ──")
print(f"  ρ = {rho:.3f},  p = {p_spear:.4f}")
print(f"  Strong {'negative' if rho < 0 else 'positive'} correlation — {'significant ✓' if p_spear < 0.05 else 'not significant'}")

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(go.Bar(
    x=monthly['month_label'], y=monthly['rate'],
    name='Subscription Rate (%)',
    marker_color='#636EFA',
    hovertemplate='<b>%{x}</b><br>Subscription rate: %{y:.1f}%<br>Calls: %{customdata:,}<extra></extra>',
    customdata=monthly['total']
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=monthly['month_label'], y=monthly['euribor'],
    name='Euribor 3M (%)',
    mode='lines+markers',
    line=dict(color='#EF553B', width=3),
    marker=dict(size=8),
    hovertemplate='<b>%{x}</b><br>Euribor 3M: %{y:.2f}%<extra></extra>'
), secondary_y=True)

fig.update_layout(
    title=dict(text=f'Subscription Rate vs Euribor 3M by Month  |  Spearman ρ = {rho:.2f}, p = {p_spear:.4f}', font_size=15),
    height=480,
    legend=dict(x=0.01, y=0.99),
    hovermode='x unified'
)
fig.update_yaxes(title_text='Subscription Rate (%)', secondary_y=False)
fig.update_yaxes(title_text='Euribor 3M Interest Rate (%)', secondary_y=True)
fig.show()

print("\nMonthly breakdown:")
print(monthly[['month_label','total','subscribed','rate','euribor']].to_string(index=False))

── Spearman Correlation: euribor3m vs monthly subscription rate ──
  ρ = -0.782,  p = 0.0075
  Strong negative correlation — significant ✓



Monthly breakdown:
month_label  total  subscribed      rate  euribor
        Mar    546         276 50.549451 1.162745
        Apr   2632         539 20.478723 1.361070
        May  13769         886  6.434745 3.293665
        Jun   5318         559 10.511470 4.256908
        Jul   7174         649  9.046557 4.685678
        Aug   6178         655 10.602137 4.300623
        Sep    570         256 44.912281 0.834760
        Oct    718         315 43.871866 1.200123
        Nov   4101         416 10.143867 3.723123
        Dec    182          89 48.901099 0.865319


---
## Section 4 — The 3D View: Age, Call Duration & Interest Rate

Three of our strongest signals — client age, call duration, and prevailing interest rate — plotted together.  
Color = subscription outcome. **Rotate the chart** to explore the structure.

Hover over any point to see job, marital status, and education.

In [9]:
# Sample for performance (5k points is plenty for a 3D scatter)
sample = df.sample(5000, random_state=42)

fig = px.scatter_3d(
    sample,
    x='age', y='duration', z='euribor3m',
    color='y',
    color_discrete_map={'yes': '#00CC96', 'no': '#EF553B'},
    opacity=0.6,
    size_max=4,
    labels={'age': 'Age', 'duration': 'Call Duration (s)', 'euribor3m': 'Euribor 3M', 'y': 'Subscribed'},
    title='3D View: Age × Call Duration × Interest Rate — colored by subscription outcome',
    hover_data={'job': True, 'marital': True, 'education': True,
                'age': True, 'duration': True, 'euribor3m': ':.2f', 'y': True}
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(
    height=600,
    scene=dict(
        xaxis_title='Age',
        yaxis_title='Call Duration (s)',
        zaxis_title='Euribor 3M (%)'
    )
)
fig.show()

---
## Section 5 — Campaign Fatigue

**Test: Kruskal-Wallis H** across campaign contact count groups  

How does subscription rate change the more times you call the same person?  
We hypothesize diminishing returns — a classic pattern in direct marketing.

In [10]:
# Cap at 10 contacts (beyond that, very small n)
df_camp = df[df['campaign'] <= 10].copy()
groups = [df_camp[df_camp['campaign'] == i]['subscribed'].values for i in range(1, 11)]

h_stat, p_kw = stats.kruskal(*groups)
print("── Kruskal-Wallis H: subscription rate across contact counts (1–10) ──")
print(f"  H = {h_stat:.2f},  p = {p_kw:.2e}")
print(f"  Result: {'Significant ✓' if p_kw < 0.05 else 'Not significant'}")

camp_stats = (
    df_camp.groupby('campaign')
    .agg(rate=('subscribed', 'mean'), n=('subscribed', 'count'))
    .reset_index()
)
camp_stats['rate_pct'] = camp_stats['rate'] * 100

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(
    x=camp_stats['campaign'], y=camp_stats['rate_pct'],
    name='Subscription Rate (%)',
    marker_color='#AB63FA',
    hovertemplate='<b>Contact #%{x}</b><br>Subscription rate: %{y:.1f}%<br>Calls made: %{customdata:,}<extra></extra>',
    customdata=camp_stats['n']
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=camp_stats['campaign'], y=camp_stats['n'],
    name='Number of Calls',
    mode='lines+markers',
    line=dict(color='#FFA15A', width=2, dash='dot'),
    hovertemplate='Contact #%{x}: %{y:,} calls<extra></extra>'
), secondary_y=True)

fig.update_layout(
    title=dict(text=f'Campaign Fatigue: Subscription Rate Drops with Each Call  |  Kruskal-Wallis p = {p_kw:.2e}', font_size=14),
    xaxis_title='Number of Contacts This Campaign',
    height=450,
    hovermode='x unified'
)
fig.update_yaxes(title_text='Subscription Rate (%)', secondary_y=False)
fig.update_yaxes(title_text='Number of Calls', secondary_y=True)
fig.show()

r1 = camp_stats.loc[camp_stats['campaign']==1,'rate_pct'].values[0]
r3 = camp_stats.loc[camp_stats['campaign']==3,'rate_pct'].values[0]
print(f"\n  Contact #1 → {r1:.1f}% subscription rate")
print(f"  Contact #3 → {r3:.1f}% subscription rate")
print(f"  Insight: Stop calling after 3 attempts — the signal is clear.")

── Kruskal-Wallis H: subscription rate across contact counts (1–10) ──
  H = 151.17,  p = 5.04e-28
  Result: Significant ✓



  Contact #1 → 13.0% subscription rate
  Contact #3 → 10.7% subscription rate
  Insight: Stop calling after 3 attempts — the signal is clear.


---
## Section 6 — Previous Campaign Outcome

**Test: Chi-Square test of independence** — `poutcome` vs `y`

If a client said yes in a *previous* campaign, does that predict success in this one?  
Intuition says yes — and the numbers confirm it dramatically.

In [11]:
contingency = pd.crosstab(df['poutcome'], df['y'])
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)

print("── Chi-Square: Previous Campaign Outcome vs Current Subscription ──")
print(f"  χ² = {chi2:.2f},  df = {dof},  p = {p_chi:.2e}")
print(f"  Result: {'Significant ✓' if p_chi < 0.05 else 'Not significant'}")
print("\n  Contingency table:")
print(contingency)

pout_rates = (
    df.groupby('poutcome')['subscribed']
    .agg(['mean','count'])
    .rename(columns={'mean':'rate','count':'n'})
    .reset_index()
)
pout_rates['rate_pct'] = pout_rates['rate'] * 100
pout_rates = pout_rates.sort_values('rate_pct', ascending=True)

fig = px.bar(
    pout_rates, x='rate_pct', y='poutcome',
    orientation='h',
    color='rate_pct',
    color_continuous_scale='Tealgrn',
    labels={'rate_pct': 'Subscription Rate (%)', 'poutcome': 'Previous Campaign Outcome'},
    title=f'Previous Campaign Outcome Strongly Predicts Current Subscription  |  χ² p = {p_chi:.2e}',
    text='rate_pct',
    hover_data={'n': True, 'rate_pct': ':.1f'}
)
fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Subscription rate: %{x:.1f}%<br>Contacts: %{customdata[0]:,}<extra></extra>'
)
fig.update_layout(height=380, coloraxis_showscale=False, xaxis_range=[0,75])
fig.show()

success_rate = pout_rates.loc[pout_rates['poutcome']=='success','rate_pct'].values[0]
nonexist_rate = pout_rates.loc[pout_rates['poutcome']=='nonexistent','rate_pct'].values[0]
print(f"\n  Previous success → {success_rate:.1f}% subscription rate")
print(f"  No prior contact → {nonexist_rate:.1f}% subscription rate")
print(f"  Uplift: {success_rate/nonexist_rate:.1f}× higher conversion from previously-won customers")

── Chi-Square: Previous Campaign Outcome vs Current Subscription ──
  χ² = 4230.52,  df = 2,  p = 0.00e+00
  Result: Significant ✓

  Contingency table:
y               no   yes
poutcome                
failure       3647   605
nonexistent  32422  3141
success        479   894



  Previous success → 65.1% subscription rate
  No prior contact → 8.8% subscription rate
  Uplift: 7.4× higher conversion from previously-won customers


---
## Summary: What We Found

| Test | Question | Result |
|------|----------|--------|
| Mann-Whitney U | Do subscribers get longer calls? | Yes — 2.7× longer median duration (p < 0.001) |
| Spearman ρ | Does a lower interest rate mean more subscriptions? | Strong negative correlation (ρ ≈ −0.77) |
| Kruskal-Wallis | Does subscription rate drop with repeated calls? | Yes — significant decline after contact #1 (p < 0.001) |
| Chi-Square | Does prior campaign success predict current? | Yes — 6.5× higher rate vs no prior contact (p < 0.001) |

### The strategic takeaway

> **Call the right people, at the right economic moment, no more than 3 times.**  
> Prioritize clients who said yes before. Time campaigns to low-euribor windows.  
> The current approach wastes ~70% of call volume on contacts #4 and beyond with sub-8% conversion.

---
*Analysis by Dany Sigha | PyData 2026 Hackathon | Dataset: UCI Bank Marketing (Moro et al., 2014)*